# Speech Coach — Open-Source Roleplay Practice (Colab Edition)

Practice spoken English through live roleplay conversations with an LLM playing the
other side (interviewer / small-talk partner / debate opponent), and get a coaching
report at the end — grammar, fluency, and an approximate pronunciation signal.

**Fully open-source speech pipeline** (STT, pronunciation scoring, TTS all run in this
Colab VM). The only thing that talks to an external service is the LLM doing the
roleplay/judging, and that's switchable between Claude's API and any OpenAI-compatible
endpoint (including plain OpenAI, if that's the paid key you'd rather use).

**Colab-specific note**: Colab has no physical microphone — recording works by asking
your *browser tab* for mic permission and streaming audio from there into this notebook.
The first time you record, your browser will show a permission popup — accept it.

**Run cells top to bottom.** Each roleplay turn re-runs the "record one turn" cell.


## 1. Install dependencies

This takes a few minutes the first time (downloads torch, transformers, whisper, piper).

In [ ]:
!pip -q install faster-whisper transformers torch phonemizer python-Levenshtein \
    anthropic openai piper-tts soundfile numpy
!apt-get -qq install -y espeak-ng > /dev/null
print("Done installing packages.")


## 2. Download a Piper voice (local TTS)

Piper development lives at [`OHF-Voice/piper1-gpl`](https://github.com/OHF-Voice/piper1-gpl)
(GPL-3.0 — the original MIT repo was archived in Oct 2025). This downloads one voice
model (~60MB) to the Colab VM's local disk.

In [ ]:
!python3 -m piper.download_voices en_US-lessac-medium
print("Voice downloaded.")


## 3. Configure your LLM provider

Pick ONE of the two cells below and run it. Keys are entered via `getpass` so they
aren't saved into the notebook file.

In [ ]:
# --- OPTION A: Claude API (pay-per-token, no subscription, best roleplay quality) ---
import os
from getpass import getpass

LLM_PROVIDER = "anthropic"
os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
CLAUDE_MODEL = "claude-sonnet-4-5"
print("Configured: Claude API")


In [ ]:
# --- OPTION B: Any OpenAI-compatible endpoint (OpenAI itself, or a hosted open model) ---
# NOTE: Colab is a remote VM, so "local" servers like Ollama on your own laptop are NOT
# reachable from here unless you tunnel them (e.g. ngrok). Pointing this at
# https://api.openai.com/v1 with an OpenAI key is the easiest fully-supported option
# if you'd rather not use Claude for this piece.
import os
from getpass import getpass

LLM_PROVIDER = "openai_compatible"
OPENAI_COMPATIBLE_BASE_URL = input("Base URL (e.g. https://api.openai.com/v1): ").strip()
OPENAI_COMPATIBLE_API_KEY = getpass("API key: ")
OPENAI_COMPATIBLE_MODEL = input("Model name (e.g. gpt-4o-mini): ").strip()
os.environ["OPENAI_COMPATIBLE_API_KEY"] = OPENAI_COMPATIBLE_API_KEY
print("Configured: OpenAI-compatible endpoint")


## 4. LLM client abstraction

Same interface regardless of which provider you configured above.

In [ ]:
class LLMClient:
    def __init__(self):
        self.provider = LLM_PROVIDER
        if self.provider == "anthropic":
            from anthropic import Anthropic
            self._client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
            self._model = CLAUDE_MODEL
        elif self.provider == "openai_compatible":
            from openai import OpenAI
            self._client = OpenAI(base_url=OPENAI_COMPATIBLE_BASE_URL, api_key=OPENAI_COMPATIBLE_API_KEY)
            self._model = OPENAI_COMPATIBLE_MODEL
        else:
            raise ValueError(f"Unknown LLM_PROVIDER '{self.provider}'")

    def chat(self, system_prompt, messages, max_tokens=500):
        if self.provider == "anthropic":
            response = self._client.messages.create(
                model=self._model, max_tokens=max_tokens,
                system=system_prompt, messages=messages,
            )
            return response.content[0].text
        else:
            full_messages = [{"role": "system", "content": system_prompt}] + messages
            response = self._client.chat.completions.create(
                model=self._model, max_tokens=max_tokens, messages=full_messages,
            )
            return response.choices[0].message.content

print("LLMClient ready.")


## 5. Roleplay scenarios

In [ ]:
SCENARIOS = {
    "interview": {
        "label": "Job Interview",
        "system_prompt": (
            "You are an experienced hiring manager conducting a spoken job interview "
            "with a candidate who is practicing their English fluency. Ask one question "
            "at a time, react naturally to their answer, and keep the conversation moving "
            "like a real interview. Stay fully in character — do not give feedback or "
            "coaching during the conversation; that happens afterward, separately. Keep "
            "responses conversational and fairly short (2-4 sentences)."
        ),
        "opening": "Let's begin. Tell me a little about yourself and why you're interested in this role.",
    },
    "small_talk": {
        "label": "Small Talk / Casual Chat",
        "system_prompt": (
            "You are a friendly stranger making casual conversation with someone practicing "
            "their spoken English. Keep it light, ask follow-up questions, share small "
            "opinions, and let the conversation wander naturally. Stay in character. Keep "
            "responses short and conversational (1-3 sentences)."
        ),
        "opening": "Hey, this line is taking forever, huh? So what brings you here today?",
    },
    "debate": {
        "label": "Friendly Debate",
        "system_prompt": (
            "You are a sharp but good-natured debate partner. Take the opposing side of "
            "whatever position the user argues, push back with real counterarguments, and "
            "ask them to justify their reasoning — but keep it friendly, not hostile. Stay "
            "in character. Keep responses conversational (2-4 sentences)."
        ),
        "opening": "Alright, give me a topic you have an opinion on, and I'll argue the other side.",
    },
}

print("Choose a scenario:")
for i, (k, v) in enumerate(SCENARIOS.items(), 1):
    print(f"  {i}. {v['label']}  (key: {k!r})")

SCENARIO_KEY = input("Enter the key (interview / small_talk / debate): ").strip()
scenario = SCENARIOS[SCENARIO_KEY]
print(f"Selected: {scenario['label']}")


## 6. Browser mic recorder

Colab has no physical mic — this grants access to your **browser tab's** mic and
records a fixed-length clip each time you call `record_audio(seconds)`. The first
call will prompt for mic permission in the browser.

In [ ]:
from google.colab import output
from IPython.display import Javascript, display
import base64, io
import numpy as np
from pydub import AudioSegment

RECORDER_JS = """
async function recordAudio(duration_ms) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const stopped = new Promise(resolve => { recorder.onstop = resolve; });
  recorder.start();
  await new Promise(resolve => setTimeout(resolve, duration_ms));
  recorder.stop();
  await stopped;
  stream.getTracks().forEach(t => t.stop());
  const blob = new Blob(chunks, { type: 'audio/webm' });
  const buffer = await blob.arrayBuffer();
  let binary = '';
  const bytes = new Uint8Array(buffer);
  for (let i = 0; i < bytes.byteLength; i++) binary += String.fromCharCode(bytes[i]);
  return btoa(binary);
}
"""

SAMPLE_RATE = 16000

def record_audio(duration=6):
    """Records `duration` seconds from the browser mic. Returns int16 mono samples at 16kHz."""
    display(Javascript(RECORDER_JS))
    print(f"Recording for {duration}s -- speak now...")
    b64 = output.eval_js(f"recordAudio({int(duration * 1000)})")
    audio_bytes = base64.b64decode(b64)
    seg = AudioSegment.from_file(io.BytesIO(audio_bytes), format="webm")
    seg = seg.set_frame_rate(SAMPLE_RATE).set_channels(1).set_sample_width(2)
    samples = np.array(seg.get_array_of_samples(), dtype=np.int16)
    print("Recording captured.")
    return samples

print("record_audio() ready. Test it in the next cell if you like.")


## 7. STT (faster-whisper) + pronunciation scoring (wav2vec2, local)

**Honest limitation**: without a fixed reference script, "expected" phonemes are derived
from what Whisper *thought* it heard. If Whisper mishears a word because it was
mispronounced, the expected phonemes shift to match the error — treat this as a rough
session-level trend, not a precise per-word grade.

In [ ]:
from faster_whisper import WhisperModel
import torch
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from phonemizer import phonemize
import Levenshtein

print("Loading Whisper model...")
whisper_model = WhisperModel("small", device="cpu", compute_type="int8")

PHONEME_MODEL_NAME = "facebook/wav2vec2-lv-60-espeak-cv-ft"
print("Loading phoneme model (this can take a minute the first time)...")
phoneme_processor = Wav2Vec2Processor.from_pretrained(PHONEME_MODEL_NAME)
phoneme_model = Wav2Vec2ForCTC.from_pretrained(PHONEME_MODEL_NAME)
phoneme_model.eval()

def transcribe(audio):
    if audio.size == 0:
        return ""
    audio_float = audio.astype(np.float32) / 32768.0
    segments, _ = whisper_model.transcribe(audio_float, language="en", beam_size=5)
    return " ".join(seg.text.strip() for seg in segments).strip()

def _recognize_phonemes(audio):
    audio_float = audio.astype(np.float32) / 32768.0
    inputs = phoneme_processor(audio_float, sampling_rate=SAMPLE_RATE, return_tensors="pt")
    with torch.no_grad():
        logits = phoneme_model(inputs.input_values).logits
    pred_ids = torch.argmax(logits, dim=-1)
    return phoneme_processor.batch_decode(pred_ids)[0]

def score_pronunciation(audio, transcript):
    if not transcript.strip() or audio.size == 0:
        return {"phoneme_accuracy": 0.0}
    recognized = _recognize_phonemes(audio)
    expected = phonemize(transcript, language="en-us", backend="espeak", strip=True)
    rec_compact = recognized.replace(" ", "")
    exp_compact = expected.replace(" ", "")
    if not exp_compact:
        return {"phoneme_accuracy": 0.0}
    dist = Levenshtein.distance(rec_compact, exp_compact)
    max_len = max(len(rec_compact), len(exp_compact), 1)
    accuracy = max(0.0, (1 - dist / max_len)) * 100
    return {"phoneme_accuracy": round(accuracy, 1)}

def speech_rate_and_fillers(transcript, duration_seconds):
    fillers = {"um", "uh", "umm", "uhh", "like", "you know", "i mean", "so yeah"}
    words = transcript.lower().split()
    word_count = len(words)
    filler_count = sum(1 for w in words if w.strip(",.?!") in fillers)
    wpm = (word_count / duration_seconds) * 60 if duration_seconds > 0 else 0.0
    return {"words_per_minute": round(wpm, 1), "filler_word_count": filler_count}

print("STT + pronunciation scoring ready.")


## 8. TTS (Piper, local)

In [ ]:
import wave, tempfile, os as _os
from piper import PiperVoice
from IPython.display import Audio, display as ipy_display

piper_voice = PiperVoice.load("en_US-lessac-medium.onnx")

def speak(text):
    """Synthesizes speech and plays it inline in the notebook output."""
    tmp_path = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    with wave.open(tmp_path, "wb") as wav_file:
        piper_voice.synthesize_wav(text, wav_file)
    ipy_display(Audio(tmp_path, autoplay=True))
    _os.remove(tmp_path)

print("speak() ready -- audio will play inline after each partner turn.")


## 9. Run the session

Each time you run the next cell, it records ~6 seconds from your mic, transcribes it,
scores pronunciation, gets the partner's reply, and speaks it back. Re-run the cell for
each turn. Say something like "let's stop here" (or just move to the report cell whenever
you're done) to end.

Adjust `duration` in `record_audio(duration=6)` if 6 seconds feels too short or too long
for your typical turn.

In [ ]:
turns = []
conversation_messages = []

llm = LLMClient()

print(f"Partner: {scenario['opening']}\n")
speak(scenario['opening'])
conversation_messages.append({"role": "assistant", "content": scenario['opening']})


### Run this cell once per turn:

In [ ]:
audio = record_audio(duration=6)
duration_seconds = audio.size / SAMPLE_RATE if audio.size else 0.0

transcript = transcribe(audio).strip()
print(f"You: {transcript}\n")

if transcript:
    pron = score_pronunciation(audio, transcript)
    fluency = speech_rate_and_fillers(transcript, duration_seconds)

    conversation_messages.append({"role": "user", "content": transcript})
    reply = llm.chat(scenario["system_prompt"], conversation_messages, max_tokens=300)
    conversation_messages.append({"role": "assistant", "content": reply})

    print(f"Partner: {reply}\n")
    speak(reply)

    turns.append({
        "transcript": transcript,
        "assistant_reply": reply,
        "phoneme_accuracy": pron["phoneme_accuracy"],
        "words_per_minute": fluency["words_per_minute"],
        "filler_word_count": fluency["filler_word_count"],
    })
else:
    print("(Didn't catch that -- try running this cell again.)")


## 10. End-of-session coaching report

Run this whenever you're done practicing.

In [ ]:
from statistics import mean
import json
from datetime import datetime

FEEDBACK_SYSTEM_PROMPT = """You are an expert spoken-English coach reviewing a practice
session transcript. You're given:
  1. The full conversation transcript (user's spoken turns + partner's replies)
  2. Aggregated local pronunciation/fluency signals per turn:
     - phoneme_accuracy: an approximate 0-100 score from comparing recognized vs.
       expected phonemes (a local, open-source approximation, not clinical-grade --
       treat it as a rough signal)
     - words_per_minute and filler_word_count per turn

Give feedback that is specific and actionable, not generic encouragement. Structure your
response as:

**Strengths** (2-3 concrete things, referencing specific moments from the transcript)

**Grammar & Phrasing Issues** (specific sentences that had errors, with the corrected version)

**Fluency Notes** (filler words, pace, hesitation patterns)

**Pronunciation** (based on the phoneme_accuracy signal -- keep this general: overall
trend, not claims about specific sounds)

**Top 3 Priorities** (ranked by impact)

**Resources** (2-4 specific, real resources matched to the priorities above)

Be direct and honest. Do not pad with excessive praise."""

def generate_report(turns):
    valid_turns = [t for t in turns if t["transcript"]]
    if not valid_turns:
        return "No speech was captured this session -- nothing to give feedback on."

    lines = []
    for i, t in enumerate(valid_turns, 1):
        lines.append(f"[Turn {i}] You: {t['transcript']}")
        lines.append(f"[Turn {i}] Partner: {t['assistant_reply']}")
    transcript_text = "\n".join(lines)

    avg_phoneme = mean(t["phoneme_accuracy"] for t in valid_turns)
    avg_wpm = mean(t["words_per_minute"] for t in valid_turns)
    total_fillers = sum(t["filler_word_count"] for t in valid_turns)

    user_content = f"""SESSION TRANSCRIPT:
{transcript_text}

AGGREGATE LOCAL SIGNALS:
- Average phoneme accuracy (approximate, 0-100): {avg_phoneme:.1f}
- Average speaking pace: {avg_wpm:.1f} words/minute
- Total filler words used across session: {total_fillers}

Please generate the coaching report."""

    return llm.chat(FEEDBACK_SYSTEM_PROMPT, [{"role": "user", "content": user_content}], max_tokens=1500)

report = generate_report(turns)
print("=" * 60)
print(report)
print("=" * 60)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = f"session_{SCENARIO_KEY}_{timestamp}.json"
with open(log_path, "w") as f:
    json.dump({"scenario": SCENARIO_KEY, "turns": turns, "report": report}, f, indent=2)
print(f"\nSession saved to {log_path} (right-click it in the Colab file browser to download -- it won't persist after the runtime disconnects).")
